In [1]:
!pip install flask twilio


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import requests
import json
from twilio.rest import Client

# --- Twilio setup variables ---
ACCOUNT_SID = "AC54f25d5f72cd43c3127c1b695b55cbdf"
AUTH_TOKEN = "bab957d2b0c26cda6308d9975f834130"
TWILIO_NUMBER = "+13509608232"
TO_NUMBER = "+32474060826"

# make client
client = Client(ACCOUNT_SID, AUTH_TOKEN)

def get_weather():
    url = "http://127.0.0.1:5001/weather/tanzania"
    response = requests.get(url)
    return response.json()

def format_message(data):
    parts = data["sms"]["message"].split(" | ")

    # --- Header ---
    header = parts[0].replace("Weather area update", "Weather")

    # --- Today ---
    today = parts[1]
    today = today.replace("Today:", "")
    today = today.replace("rain risk up to", "rain")
    today = today.strip()

    # --- Next 3 days ---
    next3 = parts[2]
    next3 = next3.replace("Next 3d:", "")
    next3 = next3.replace("Slight showers", "showers")
    next3 = next3.replace("rain up to", "")
    next3 = next3.strip()

    # --- Next 7 days ---
    next7 = parts[3]
    next7 = next7.replace("Next 7d:", "")
    next7 = next7.replace("Thunderstorm", "storm")
    next7 = next7.replace("rain up to", "")
    next7 = next7.strip()

    # --- Final compact SMS ---
    sms = f"{header} | Today: {today} | 3d: {next3} | 7d: {next7}"

    return sms


# --- 3. Send SMS ---
def send_sms(message):
    client.messages.create(
        body=message,
        from_=TWILIO_NUMBER,
        to=TO_NUMBER
    )


weather = get_weather()
print(weather)
sms_text = format_message(weather)
print("Sending:", sms_text)
send_sms(sms_text)

{'area_summary': {'next_3_days': {'dominant_weather': 'Slight showers', 'max_total_precipitation': 19.9, 'temp_max': 35.9, 'temp_min': 17.2}, 'next_7_days': {'dominant_weather': 'Thunderstorm', 'max_total_precipitation': 76.0, 'temp_max': 36.1, 'temp_min': 17.2}, 'today': {'dominant_weather': 'Overcast', 'max_rain_risk_next_12h': 40, 'max_wind_peak_next_12h': 12.7, 'temp_max': 35.7, 'temp_min': 17.5}}, 'bounding_box': {'max_lat': -0.9853, 'max_lon': 40.4432, 'min_lat': -11.7613, 'min_lon': 29.3272}, 'input_points': [], 'meta': {'generated_at': '2026-03-17T11:36:08', 'sample_count': 5, 'scope': 'area', 'source': 'open-meteo', 'timezone': 'Africa/Dar_es_Salaam'}, 'sample_points': [{'daily_summary_next_3d': {'days_considered': 3, 'dominant_weather': 'Thunderstorm + hail', 'period_end': '2026-03-19', 'period_start': '2026-03-17', 'temp_max': 29.0, 'temp_min': 17.2, 'total_precipitation': 12.0}, 'daily_summary_next_7d': {'days_considered': 7, 'dominant_weather': 'Thunderstorm', 'period_end'